# Случайный поиск для четырёх функций

В ноутбуке реализованы три алгоритма случайного поиска для функций:

1. $J_1(x, y) = x^2 + y^2$
2. $J_2(x, y) = x^2/4 + y^2/9$
3. $J_3(x, y) = 70(x-1)^2 + (y-1)^2 + 1$
4. $J_4(x, y) = (x-8)^2 + (y-1)^2 + 70\left(y + (x-8)^2 - 1\right)^2 + 1$

Реализованы методы:

- алгоритм с возвратом при неудачном шаге;
- алгоритм наилучшей пробы;
- алгоритм статистического градиента.

Также строятся траектории движения точек на плоскости $XY$, все графики сохраняются в папку `graphs`, а в конце выводится таблица с найденными минимумами.

## Допущения

Так как в задании не зафиксированы точные параметры методов, используются разумные настройки:

- у методов случайного поиска шаг постепенно уменьшается после серии неудачных попыток;
- в методе наилучшей пробы на каждой итерации генерируется несколько случайных направлений, после чего выбирается лучшая точка;
- в методе статистического градиента градиент оценивается методом симметричных конечных разностей по случайным направлениям;
- для каждой функции выбрана стартовая точка, удалённая от минимума, чтобы траектория была наглядной.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (6.5, 5.2)
plt.rcParams["figure.dpi"] = 140

BASE_DIR = Path.cwd()
GRAPHS_DIR = BASE_DIR / "graphs"
GRAPHS_DIR.mkdir(exist_ok=True)

In [ ]:
def f1(v):
    x, y = v
    return x**2 + y**2

def f2(v):
    x, y = v
    return x**2 / 4 + y**2 / 9

def f3(v):
    x, y = v
    return 70 * (x - 1)**2 + (y - 1)**2 + 1

def f4(v):
    x, y = v
    return (x - 8)**2 + (y - 1)**2 + 70 * (y + (x - 8)**2 - 1)**2 + 1

functions = {
    "f1": {
        "func": f1,
        "title": r"$J(x,y)=x^2+y^2$",
        "start": np.array([6.0, -5.0]),
        "xlim": (-7, 7),
        "ylim": (-7, 7),
        "true_min": np.array([0.0, 0.0]),
    },
    "f2": {
        "func": f2,
        "title": r"$J(x,y)=x^2/4+y^2/9$",
        "start": np.array([7.0, -6.0]),
        "xlim": (-8, 8),
        "ylim": (-8, 8),
        "true_min": np.array([0.0, 0.0]),
    },
    "f3": {
        "func": f3,
        "title": r"$J(x,y)=70(x-1)^2+(y-1)^2+1$",
        "start": np.array([-1.5, 7.0]),
        "xlim": (-3, 4),
        "ylim": (-1, 8),
        "true_min": np.array([1.0, 1.0]),
    },
    "f4": {
        "func": f4,
        "title": r"$J(x,y)=(x-8)^2+(y-1)^2+70(y+(x-8)^2-1)^2+1$",
        "start": np.array([4.0, 5.0]),
        "xlim": (3, 10),
        "ylim": (-1, 8),
        "true_min": np.array([8.0, 1.0]),
    },
}

In [ ]:
def random_unit_2d(rng):
    v = rng.normal(size=2)
    return v / np.linalg.norm(v)

In [ ]:
def search_with_return(func, x0, alpha=0.35, n_iter=120, tol=1e-7):
    rng = np.random.default_rng()
    x = np.array(x0, dtype=float)
    fx = func(x)
    traj = [x.copy()]
    no_improve = 0
    step = alpha

    for _ in range(n_iter):
        u = random_unit_2d(rng)
        candidate = x + step * u
        fc = func(candidate)

        if fc < fx:
            x, fx = candidate, fc
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 10:
                step *= 0.7
                no_improve = 0

        traj.append(x.copy())

        if step < tol:
            break

    return np.array(traj), x, fx

In [ ]:
def best_probe(func, x0, alpha=0.4, n_iter=100, probes=25, tol=1e-7):
    rng = np.random.default_rng()
    x = np.array(x0, dtype=float)
    fx = func(x)
    traj = [x.copy()]
    step = alpha
    fail_rounds = 0

    for _ in range(n_iter):
        candidates = np.array([x + step * random_unit_2d(rng) for _ in range(probes)])
        values = np.array([func(c) for c in candidates])
        best_idx = np.argmin(values)

        if values[best_idx] < fx:
            x, fx = candidates[best_idx], values[best_idx]
            fail_rounds = 0
        else:
            fail_rounds += 1
            if fail_rounds >= 5:
                step *= 0.75
                fail_rounds = 0

        traj.append(x.copy())

        if step < tol:
            break

    return np.array(traj), x, fx

In [ ]:
def statistical_gradient(func, x0, eta=0.08, mu=0.25, n_iter=120, probes=40, grad_tol=1e-6):
    rng = np.random.default_rng()
    x = np.array(x0, dtype=float)
    traj = [x.copy()]

    for k in range(n_iter):
        directions = np.array([random_unit_2d(rng) for _ in range(probes)])
        diffs = np.array([
            (func(x + mu * u) - func(x - mu * u)) / (2 * mu)
            for u in directions
        ])
        grad_est = 2.0 * (diffs[:, None] * directions).mean(axis=0)

        if np.linalg.norm(grad_est) < grad_tol:
            break

        step = eta / (1.0 + 0.02 * k)
        x = x - step * grad_est
        traj.append(x.copy())

    return np.array(traj), x, func(x)

In [ ]:
algorithms = {
    "return": {
        "runner": search_with_return,
        "title": "С возвратом при неудачном шаге",
    },
    "best_probe": {
        "runner": best_probe,
        "title": "Наилучшей пробы",
    },
    "stat_grad": {
        "runner": statistical_gradient,
        "title": "Статистического градиента",
    },
}

In [ ]:
results = {}

for fname, finfo in functions.items():
    results[fname] = {}
    for aname, ainfo in algorithms.items():
        traj, xmin, fmin = ainfo["runner"](finfo["func"], finfo["start"])
        results[fname][aname] = {
            "traj": traj,
            "xmin": xmin,
            "fmin": fmin,
        }

In [ ]:
def save_trajectory_plot(func, title, xlim, ylim, traj, found_min, true_min, method_title, save_path):
    xs = np.linspace(xlim[0], xlim[1], 300)
    ys = np.linspace(ylim[0], ylim[1], 300)
    X, Y = np.meshgrid(xs, ys)
    Z = np.vectorize(lambda a, b: func(np.array([a, b])))(X, Y)

    plt.figure()
    try:
        levels = np.geomspace(max(Z.min() + 1e-6, 1e-6), Z.max(), 25)
        plt.contour(X, Y, Z, levels=levels, linewidths=0.8)
    except Exception:
        plt.contour(X, Y, Z, levels=25, linewidths=0.8)

    plt.plot(traj[:, 0], traj[:, 1], marker="o", markersize=2.5, linewidth=1.4, label="Траектория")
    plt.scatter([traj[0, 0]], [traj[0, 1]], marker="s", s=55, label="Старт")
    plt.scatter([found_min[0]], [found_min[1]], marker="*", s=110, label="Найденный минимум")
    plt.scatter([true_min[0]], [true_min[1]], marker="x", s=70, label="Истинный минимум")

    plt.title(f"{title}\nМетод: {method_title}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
summary_rows = []

for fname, finfo in functions.items():
    for aname, ainfo in algorithms.items():
        traj = results[fname][aname]["traj"]
        xmin = results[fname][aname]["xmin"]
        fmin = results[fname][aname]["fmin"]

        save_path = GRAPHS_DIR / f"{fname}_{aname}.png"
        save_trajectory_plot(
            func=finfo["func"],
            title=finfo["title"],
            xlim=finfo["xlim"],
            ylim=finfo["ylim"],
            traj=traj,
            found_min=xmin,
            true_min=finfo["true_min"],
            method_title=ainfo["title"],
            save_path=save_path,
        )

        summary_rows.append({
            "Функция": fname,
            "Метод": ainfo["title"],
            "x_min": xmin[0],
            "y_min": xmin[1],
            "J(x_min, y_min)": fmin,
            "Число шагов": len(traj) - 1,
            "Файл графика": save_path.name,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
summary_df_rounded = summary_df.copy()
for col in ["x_min", "y_min", "J(x_min, y_min)"]:
    summary_df_rounded[col] = summary_df_rounded[col].map(lambda v: round(float(v), 6))

summary_df_rounded

In [ ]:
summary_df_rounded.to_csv(BASE_DIR / "results.csv", index=False)
print(f"Графики сохранены в: {GRAPHS_DIR.resolve()}")
print(f"Таблица результатов сохранена в: {(BASE_DIR / 'results.csv').resolve()}")